In [ ]:
import numpy as np
import plotly
from numpy.typing import ArrayLike

# Binary variables

In [2]:
class BinaryPairwiseBettingMartingale():

    def __init__(self):
        self.martingale = list()
        self.observed = {0: 0, 1: 0 }
        self.transitions = {0: 0, 1: 0}
        self.last_observed = None

    @property 
    def parameters(self):
        if sum(list(self.observed.values())) < 2 :
            return None

        params = {}
        for key in [0, 1]:
            params[key] = self.transitions[key] / self.observed[key]
        return params

    @staticmethod
    def validate_sequence_shape(X: ArrayLike):
        if not (len(X.shape) == 1):
            raise ValueError("Sequence must be a one dimensionnal iterable")

    @staticmethod
    def different_values(X1, X2):
        return X1 != X2

    def update_parameters(self, X1, X2):
        self.observed[X1] += 1
        if self.different_values(X1, X2):
            self.transitions[X1] += 1
        return self
    
    def null_hypothesis_likelihood(self, X1, X2):
        return 1/2

    def alternative_hypothesis_likelihood(self, X1, X2):
        params = self.parameters
        diff_last_X1 = self.different_values(self.last_observed, X1)
        diff_last_X2 = self.different_values(self.last_observed, X2)
        
        num = diff_last_X1 * params[self.last_observed] + (1 - diff_last_X1) * (1 - params[self.last_observed])
        num *= params[X1]
        denum = diff_last_X2 * params[self.last_observed] + (1 - diff_last_X2) * (1 - params[self.last_observed])
        denum *= params[X2]
        denum += num

        return num / denum


    def bet(self, X1, X2):
        if self.different_values(X1, X2) and len(self.martingale) > 2:
            return ( 
                self.alternative_hypothesis_likelihood(X1, X2) /
                self.null_hypothesis_likelihood(X1, X2)
            )
        return 1

    def run_martingale(self, X:ArrayLike):
        self.validate_sequence_shape(X)

        X1 = self.last_observed
        for i in np.arange(0, len(X), step=2):
            X2 = X[i]
            if i > 0 :
                X1 = X[i-1]
            if X1 is not None :
                m = self.martingale[-1] * self.bet(X1, X2)
                self.update_parameters(self.last_observed, X1)
                self.update_parameters(X1, X2)
            else :
                m = 1
            self.martingale.append(m)
            self.last_observed = X2
        return self

In [ ]:
test_Bern = np.random.binomial(1, .5, 3000)
M_Bern = BinaryPairwiseBettingMartingale()
M_Bern = M.run_martingale(test)
res = np.log(M_Bern.martingale)
print(res.min(), res.max())


-0.6076446136823718 0.08807197276211452


In [18]:
test_Markov = [1]
for i in range(2999):
    current = test_Markov[i]
    if current == 1 :
        test_Markov.append(np.random.binomial(1, .2, 1)[0])
    else:
        test_Markov.append(np.random.binomial(1, .7, 1)[0])
test_Markov = np.array(test_Markov)

M_Markov = BinaryPairwiseBettingMartingale()
M_Markov = M_Markov.run_martingale(test_Markov)
res = np.log(M_Markov.martingale)
print(res.min(), res.max())

-1.134675616879781 149.95660968247327


In [25]:
res

array([  0.        ,   0.        ,   0.        , ..., 149.04475239,
       149.5005797 , 149.95660968])

# Continuous variables